In [51]:
from pathlib import Path
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD

FIG_DIR = ROOT / "reports" / "figs"
TAB_DIR = ROOT / "reports" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name: str, dpi=150):
    out = FIG_DIR / name
    fig.savefig(out, bbox_inches="tight", dpi=dpi)
    print(f"[saved] {out}")

def save_table(df, name: str, index=True):
    out = TAB_DIR / name
    df.to_csv(out, index=index)
    print(f"[saved] {out}")

In [52]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
import sys; sys.path.append("src")

plt.rcParams["figure.figsize"] = (10, 6); sns.set_theme(style="whitegrid")

X = pd.read_csv(
    "../data/processed/dataset.csv",
     encoding="utf-8-sig",   
    sep=";",             
    decimal=",",        
    low_memory=False
)



In [53]:
meta = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "non_null": X.notna().sum(),
    "nulls": X.isna().sum(),
    "null_%": X.isna().mean().round(3)
}).sort_values("null_%", ascending=False)
meta.head(20)

,dtype,non_null,nulls,null_%
Исход,object,191,0,0.0
ЭХО ЛП перед ТП,int64,191,0,0.0
relative risk,float64,191,0,0.0
healthy person risk,float64,191,0,0.0
QRISK3,float64,191,0,0.0
ДАД перед ТП,int64,191,0,0.0
САД перед ТП,int64,191,0,0.0
ОТТ перед ТП,float64,191,0,0.0
ЭХО ИММЛЖ перед ТП,float64,191,0,0.0
ЭХО ММЛЖ перед ТП,float64,191,0,0.0


In [54]:
missing_tbl = X.isna().mean().rename("missing_rate").sort_values(ascending=False).to_frame()
save_table(missing_tbl, "missing_values.csv")

plt.figure(figsize=(12,6))
sns.heatmap(X.isna(), cbar=False)
plt.title("Missingness heatmap (X only)")
save_fig(plt.gcf(), "eda_missing_heatmap.png"); plt.close()

[saved] c:\Users\Greta\OneDrive\Рабочий стол\Документы\7 sem\diploma\reports\tables\missing_values.csv
[saved] c:\Users\Greta\OneDrive\Рабочий стол\Документы\7 sem\diploma\reports\figs\eda_missing_heatmap.png


In [55]:
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
len(num_cols), num_cols[:10]

(20,
 ['ОХ перед ТП',
  'ЛПНП перед ТП',
  'ЛПВП перед ТП',
  'ТГ перед ТП',
  'Мочевая кислота перед ТП',
  'ЭХО ЛП перед ТП',
  'ЭХО КДР перед ТП',
  'ЭХО МЖП перед ТП',
  'ЭХО ЗС перед ТП',
  'ЭХО СДЛА перед ТП'])

In [56]:
for col in num_cols:
    fig, ax = plt.subplots(1, 2, figsize=(14,4))
    X[col].hist(ax=ax[0], bins=30); ax[0].set_title(f"Hist: {col}")
    sns.boxplot(x=X[col], ax=ax[1]); ax[1].set_title(f"Box: {col}")
    save_fig(fig, f"eda_hist_{col}.png"); plt.close(fig)

[saved] c:\Users\Greta\OneDrive\Рабочий стол\Документы\7 sem\diploma\reports\figs\eda_hist_ОХ перед ТП.png
[saved] c:\Users\Greta\OneDrive\Рабочий стол\Документы\7 sem\diploma\reports\figs\eda_hist_ЛПНП перед ТП.png
[saved] c:\Users\Greta\OneDrive\Рабочий стол\Документы\7 sem\diploma\reports\figs\eda_hist_ЛПВП перед ТП.png
[saved] c:\Users\Greta\OneDrive\Рабочий стол\Документы\7 sem\diploma\reports\figs\eda_hist_ТГ перед ТП.png
[saved] c:\Users\Greta\OneDrive\Рабочий стол\Документы\7 sem\diploma\reports\figs\eda_hist_Мочевая кислота перед ТП.png
[saved] c:\Users\Greta\OneDrive\Рабочий стол\Документы\7 sem\diploma\reports\figs\eda_hist_ЭХО ЛП перед ТП.png
[saved] c:\Users\Greta\OneDrive\Рабочий стол\Документы\7 sem\diploma\reports\figs\eda_hist_ЭХО КДР перед ТП.png
[saved] c:\Users\Greta\OneDrive\Рабочий стол\Документы\7 sem\diploma\reports\figs\eda_hist_ЭХО МЖП перед ТП.png
[saved] c:\Users\Greta\OneDrive\Рабочий стол\Документы\7 sem\diploma\reports\figs\eda_hist_ЭХО ЗС перед ТП.png
[s

In [57]:
corr = X[num_cols].corr(method="pearson")
import numpy as np
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(12,10))
sns.heatmap(corr, mask=mask, cmap="coolwarm", vmin=-1, vmax=1, center=0)
plt.title("Feature–Feature Correlation (Pearson)")
save_fig(plt.gcf(), "eda_corr_heatmap.png"); plt.close()

[saved] c:\Users\Greta\OneDrive\Рабочий стол\Документы\7 sem\diploma\reports\figs\eda_corr_heatmap.png


In [58]:
pairs = []
for i, a in enumerate(num_cols):
    for b in num_cols[i+1:]:
        r = corr.loc[a, b]
        if pd.notna(r) and abs(r) > 0.7:
            pairs.append((a, b, float(r)))
high_pairs = (pd.DataFrame(pairs, columns=["feature_1","feature_2","pearson_r"])
              .sort_values("pearson_r", key=lambda s: s.abs(), ascending=False))
save_table(high_pairs, "high_corr_pairs.csv")
high_pairs.head(20)

[saved] c:\Users\Greta\OneDrive\Рабочий стол\Документы\7 sem\diploma\reports\tables\high_corr_pairs.csv


,feature_1,feature_2,pearson_r
7,ЭХО ММЛЖ перед ТП,ЭХО ИММЛЖ перед ТП,0.938168
0,ЭХО КДР перед ТП,ЭХО ММЛЖ перед ТП,0.851601
9,QRISK3,qrisk age,0.822502
3,ЭХО МЖП перед ТП,ЭХО ММЛЖ перед ТП,0.802918
4,ЭХО МЖП перед ТП,ЭХО ИММЛЖ перед ТП,0.769349
1,ЭХО КДР перед ТП,ЭХО ИММЛЖ перед ТП,0.768055
5,ЭХО ЗС перед ТП,ЭХО ММЛЖ перед ТП,0.758543
2,ЭХО МЖП перед ТП,ЭХО ЗС перед ТП,0.755662
8,САД перед ТП,ДАД перед ТП,0.747620
6,ЭХО ЗС перед ТП,ЭХО ИММЛЖ перед ТП,0.737857


In [59]:
def tukey_outlier_mask(s: pd.Series, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    lo, hi = q1 - k*iqr, q3 + k*iqr
    return (s < lo) | (s > hi), (lo, hi)

out_rows = []
for col in num_cols:
    s = X[col].dropna()
    if s.empty: continue
    mask, (lo, hi) = tukey_outlier_mask(s)
    out_rows.append({"feature": col, "outliers_%": float(mask.mean()*100), "lo_cap": lo, "hi_cap": hi})
out_summary = pd.DataFrame(out_rows).sort_values("outliers_%", ascending=False)
out_summary.head(20)

,feature,outliers_%,lo_cap,hi_cap
9,ЭХО СДЛА перед ТП,37.172775,30.0000,30.0000
10,ЭХО ФВ перед ТП,31.937173,56.0000,56.0000
18,relative risk,11.518325,-21.7500,54.6500
12,ЭХО ИММЛЖ перед ТП,4.712042,23.7650,211.9650
7,ЭХО МЖП перед ТП,4.712042,7.0000,15.0000
14,САД перед ТП,4.712042,115.0000,235.0000
11,ЭХО ММЛЖ перед ТП,4.188482,36.5300,391.9700
3,ТГ перед ТП,3.141361,-0.8000,4.8000
5,ЭХО ЛП перед ТП,3.141361,24.2500,50.2500
15,ДАД перед ТП,3.141361,60.0000,140.0000


In [60]:
import pandas as pd, numpy as np

N, P = X.shape
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

miss = X.isna().mean()
print("Строк:", N)
print("Признаков всего:", P)
print("Числовых:", len(num_cols), "Категориальных:", len(cat_cols))
print("Медиана пропусков, %:", round(miss.median()*100, 2))
print("Максимум пропусков, %:", round(miss.max()*100, 2), "в", miss.sort_values(ascending=False).index[0])

print("\nТоп-10 по пропускам:")
print((miss.sort_values(ascending=False).head(10)*100).round(2))

print("\nКардинальности категориальных (топ-10):")
if cat_cols:
    print(X[cat_cols].nunique(dropna=False).sort_values(ascending=False).head(10))
else:
    print("—")

# если есть high_pairs DataFrame:
try:
    from pathlib import Path
    hp = pd.read_csv(Path("reports/tables/high_corr_pairs.csv"))
    print("\nВысокие корреляции (топ-10 по |r|):")
    print(hp.reindex(hp['pearson_r'].abs().sort_values(ascending=False).index).head(10))
except Exception:
    pass


Строк: 191
Признаков всего: 31
Числовых: 20 Категориальных: 11
Медиана пропусков, %: 0.0
Максимум пропусков, %: 0.0 в Исход

Топ-10 по пропускам:
Исход                  0.0
ЭХО ЛП перед ТП        0.0
relative risk          0.0
healthy person risk    0.0
QRISK3                 0.0
ДАД перед ТП           0.0
САД перед ТП           0.0
ОТТ перед ТП           0.0
ЭХО ИММЛЖ перед ТП     0.0
ЭХО ММЛЖ перед ТП      0.0
dtype: float64

Кардинальности категориальных (топ-10):
Диагноз                10
стадия ХСН перед ТП     4
Донор                   3
Исход                   2
ИБС до ТП               2
ХНС до ТП               2
ОНМК до ТП              2
ТЭЛА до ТП              2
ИМ до ТП                2
КАГ до ТП               2
dtype: int64
